# 02 · Experten + Gating (Track B: SQI-gated Mixture of Experts)

**Bachelorarbeit: AF-Detektion in kontaktlosen Signalen · Nik Büttner · RWTH Aachen**

Dieses Notebook setzt die von Blaß gewünschte Architektur zusammen und wertet sie aus:

1. **Drei Experten** (cECG / PPG / BCG) liefern je eine fensterweise AF-Wahrscheinlichkeit.
2. Das **Gating-Netz** sagt aus den SQIs den Zuverlässigkeits-Fehler jeder Modalität
   vorher (Ziel aus dem GT-EKG, Bachelet-Stil) und macht daraus Gewichte.
3. **Gewichtete Fusion** → fensterweise AF-Entscheidung.

Alles läuft **leckagefrei**: patientenweise LOPO-CV (kein Patient in Train *und* Test),
Out-of-Fold-Experten-Outputs fürs Gate, und das GT-EKG fließt **nur** ins Trainings-Ziel
des Gates — nie in den Test-Pfad.

> **Voraussetzung:** `01_features_sqi.ipynb` wurde gelaufen (Feature-Tabelle liegt im
> `data/`-Ordner). Für die Reliability-Tabelle wird `neurokit2` benötigt (GT-R-Zacken);
> ohne neurokit2 greift ein einfacher Fallback-Detektor.


## 1 · Imports, Pfade, Feature-Tabelle laden

In [1]:
import os, sys, glob
import numpy as np, pandas as pd

NB_DIR  = os.path.abspath('')
SRC_DIR = os.path.abspath(os.path.join(NB_DIR, '..', 'src'))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import extract as E
import experts as X
import reliability as R
import gating as G
import fusion_cv as CV
import models as M

DATA_ROOT = '../data/patients/'
DATA_DIR  = '../results/'

# Feature-Tabelle aus Notebook 01 (jüngste features_sqi_*.csv nehmen)
feat_files = sorted(glob.glob(os.path.join(DATA_DIR, 'features_sqi_*.csv')), key=os.path.getmtime)
assert feat_files, 'Keine Feature-Tabelle gefunden — erst 01_features_sqi.ipynb laufen lassen.'
df = pd.read_csv(feat_files[-1])
df, y, groups = E.split_Xygroups(df)
print('Features:', df.shape, '·', feat_files[-1])

Features: (4724, 170) · ../results\features_sqi_cecg_cwt_d2721c.csv


## 2 · Reliability-Tabelle (GT-EKG-basiertes Gate-Ziel)

Berechnet je Fenster, wie treu jede Modalität den wahren Rhythmus abbildet. Das ist
das **Trainings-Ziel** des Gates (nur hier wird das GT-EKG genutzt). Wird gecached.

`target_metric='cosen_err'` ist das AF-passende Default-Ziel (Fehler im stärksten
AF-Diskriminator). Die Tabelle enthält zusätzlich `hr_err` (Bachelet-Vergleich) und
`drr_sd_err`, sodass das Ziel später ohne Neuberechnung umgeschaltet werden kann.

In [5]:
REL_CACHE = os.path.join(DATA_DIR, 'reliability_cosen.csv')

if os.path.exists(REL_CACHE):
    rel = pd.read_csv(REL_CACHE)
    print('Reliability-Cache geladen:', rel.shape)
else:
    rel = R.build_reliability_table(DATA_ROOT, target_metric='cosen_err', n_jobs=8)
    rel.to_csv(REL_CACHE, index=False)
    print('Reliability gespeichert:', REL_CACHE)

# Abdeckung: wie oft liefert jede Modalität überhaupt einen Rhythmus?
for m in ['cecg', 'ppg', 'bcg']:
    print(f'  {m}: gültig in {rel[f"rel_{m}_valid"].mean()*100:.0f}% der Fenster')
rel.head(3)

  PAT001: 119 Fenster
  PAT003: 87 Fenster
  PAT004: 119 Fenster
  PAT005: 119 Fenster
  PAT006: 96 Fenster
  PAT007: 119 Fenster
  PAT008: 119 Fenster
  PAT009: 119 Fenster
  PAT010: 119 Fenster
  PAT011: 119 Fenster
  PAT012: 119 Fenster
  PAT013: 119 Fenster
  PAT014: 119 Fenster
  PAT015: 119 Fenster
  PAT016: 119 Fenster
  PAT017: 119 Fenster
  PAT018: 119 Fenster
  PAT019: 119 Fenster
  PAT021: 119 Fenster
  PAT022: 119 Fenster
  PAT023: 119 Fenster
  PAT024: 119 Fenster
  PAT025: 119 Fenster
  PAT026: 119 Fenster
  PAT028: 119 Fenster
  PAT029: 115 Fenster
  PAT030: 119 Fenster
  PAT031: 119 Fenster
  PAT033: 119 Fenster
  PAT034: 119 Fenster
  PAT036: 119 Fenster
  PAT037: 119 Fenster
  PAT038: 119 Fenster
  PAT039: 119 Fenster
  PAT040: 119 Fenster
  PAT041: 119 Fenster
  PAT042: 119 Fenster
  PAT043: 119 Fenster
  PAT044: 119 Fenster
  PAT045: 119 Fenster


NameError: name 'pd' is not defined

## 3 · Experten solo (Sanity + Modalitäts-Story)

Out-of-Fold-AUC je Experte. Erwartung aus deinen bisherigen Ergebnissen: PPG trägt
das Signal, cECG mittel, BCG schwach. Das ist genau die Asymmetrie, die das Gate
ausnutzen soll.

In [ ]:
from sklearn.metrics import roc_auc_score
oof = X.oof_expert_probs(df, y, groups, n_splits=5)
print('Out-of-Fold-AUC je Experte:')
for m in G.ORDER:
    print(f'  {m:5s}: {roc_auc_score(y, oof[f"p_{m}"]):.3f}')

## 4 · Hauptergebnis: gelerntes Gate vs. naive Fusion

Die zentrale Frage der Arbeit: bringt das datengetriebene, SQI-gesteuerte Gating
etwas gegenüber gleichgewichteter Fusion? `equal` = Baseline, `ridge/gb/mlp` =
gelernte Gates. Auswertung fensterweise in patientenweiser LOPO-CV.

In [ ]:
tab = CV.compare_gates(df, rel, y, groups,
                       gate_kinds=('equal', 'ridge', 'gb', 'mlp'),
                       inner_splits=5, min_spec=0.80)
cols = ['gate', 'AUC', 'Sensitivität', 'Spezifität', 'Accuracy', 'threshold']
tab[cols].round(3)

## 5 · Sensitivität des Gate-Ziels (CoSEn vs. HR vs. dRR-SD)

Welche Definition von „Zuverlässigkeit" trainiert das beste Gate? Dieselbe
Reliability-Tabelle, nur anderes Ziel — kein Neuberechnen nötig. `hr_err` entspricht
Bachelets HR-Fehler, `cosen_err`/`drr_sd_err` sind die AF-Rhythmus-Varianten.

In [ ]:
rows = []
for tm in ['cosen_err', 'drr_sd_err', 'hr_err']:
    m, t = CV.evaluate_moe(df, rel, y, groups, gate_kind='gb',
                           target_metric=tm, inner_splits=5)
    rows.append({'target': tm, **{k: m[k] for k in ['AUC','Sensitivität','Spezifität','Accuracy']}})
pd.DataFrame(rows).round(3)

## 6 · Konfidenzintervalle der besten Konfiguration

Patienten-Cluster-Bootstrap (hält alle Fenster eines Patienten zusammen) — konsistent
zur Auswertung deiner bisherigen `window_nested`-Ergebnisse.

In [ ]:
BEST_GATE   = 'gb'          # nach Tabelle in Abschnitt 4 wählen
BEST_TARGET = 'cosen_err'   # nach Tabelle in Abschnitt 5 wählen

m, t, yt, yp, yd, yg = CV.evaluate_moe(
    df, rel, y, groups, gate_kind=BEST_GATE, target_metric=BEST_TARGET,
    inner_splits=5, min_spec=0.80, return_arrays=True)

ci = M.bootstrap_ci(yt, yp, t, n_boot=2000, groups=yg)
print(f'Beste Konfiguration: gate={BEST_GATE}, target={BEST_TARGET}, threshold={t:.3f}\n')
for k in ['AUC', 'Sensitivität', 'Spezifität', 'Accuracy']:
    lo, hi = ci[k]
    print(f'  {k:12s}: {m[k]:.3f}  [{lo:.3f}, {hi:.3f}]')

## Nächste Schritte

- **Klassifikator je Experte** variieren: `clf_per_modality={'cecg':'RF','ppg':'LR','bcg':'GB'}`
  an `compare_gates`/`evaluate_moe` übergeben.
- **Gate als echtes KNN** (PyTorch + Optuna) wie bei Bachelet — der sklearn-`mlp`
  ist der Einstieg; die Schnittstelle (`gating.make_gate`) bleibt gleich.
- **Erklärbarkeit**: zeigen, wovon das Gate seine Gewichte ableitet (welche SQIs
  treiben das vorhergesagte Vertrauen?) — Integrated Gradients / Permutation Importance.

**Mit Blaß bestätigen:**
1. Gate-Ziel = RR-Irregularitäts-Treue (CoSEn), nicht reiner HR-Fehler — passt das zu AF?
2. Kanal-Aggregation PPG/BCG = bester Kanal (kleinster Fehler) — oder Mittelwert?
3. Experten eingefroren, Gate darüber trainiert (B/Stacking) — wie besprochen.
